# Final project: a taxi lakehouse with Delta Lake

**Time:** about 4 hours.

**Checklist topics this project uses:**<br><span style="color:#d93025"><b>2.x File formats</b></span><br><span style="color:#d93025"><b>3.x Joins and shuffle</b></span><br><span style="color:#d93025"><b>4.x Data skew</b></span><br><span style="color:#d93025"><b>6.x Best practices</b></span><br><span style="color:#d93025"><b>7.x Delta Lake</b></span>

## The project
You are the data engineer for a taxi analytics team. Raw trip files arrive once a month. Build a small **lakehouse** with Delta Lake that turns them into a clean table and a daily report, and keeps the report up to date when the next month arrives:

```
data/*.parquet ──► bronze (raw, append-only) ──► silver (clean, unique trips) ──► gold (daily report per borough)
                                                   └──► quarantine (rejected rows + reason)
```

There is no skeleton: write every function yourself, in the empty cells. Each milestone lists the exact rules; the check cell under it inspects your tables. Use what Labs 01–09 taught you, and the linked docs. Expect about 4 hours; it is fine to finish over the weekend. You need about 8 GB of free RAM and 3 GB of disk.

**Data:** NYC Yellow Taxi trips for January, February and March 2024 (9.5 million rows, same 19 columns as in Lab 01) and the taxi zone lookup. The setup cell downloads them and loads *zones*.

## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [Delta: write to a table](https://docs.delta.io/latest/delta-batch.html#write-to-a-table)
* [DeltaTable.isDeltaTable](https://docs.delta.io/latest/api/python/spark/index.html)
* [Delta: MERGE](https://docs.delta.io/latest/delta-update.html#upsert-into-a-table-using-merge)
* [Change data feed](https://docs.delta.io/latest/delta-change-data-feed.html)
* [functions.sha2](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.sha2.html)
* [functions.broadcast](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.broadcast.html)
* [DeltaTable.history](https://docs.delta.io/latest/delta-utility.html#retrieve-delta-table-history)
* [Delta: optimizations](https://docs.delta.io/latest/optimizations-oss.html)
* [Read the change feed](https://docs.delta.io/latest/delta-change-data-feed.html#read-changes-in-batch-queries)
* [Delta: VACUUM](https://docs.delta.io/latest/delta-utility.html#remove-files-no-longer-referenced-by-a-delta-table)
* [AQE skew join](https://spark.apache.org/docs/latest/sql-performance-tuning.html#optimizing-skew-join)

Labs 01–09 and their documentation pages are your other reference.

## Setup

In [ ]:
# Setup: run once. Downloads 3 months of trips, starts Spark with Delta Lake, defines helpers.
import hashlib, json, os, re, shutil, sys, time, urllib.request, contextlib, io
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
from pyspark.sql import SparkSession, Window, functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

MONTHS = ["2024-01", "2024-02", "2024-03"]
os.makedirs("data", exist_ok=True)
files = {f"yellow_tripdata_{m}.parquet": f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{m}.parquet" for m in MONTHS}
files["taxi_zone_lookup.csv"] = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
for name, url in files.items():
    if not os.path.exists(f"data/{name}"):
        print("downloading", name); urllib.request.urlretrieve(url, f"data/{name}")

FRESH_START = True          # deletes lake/ so every run starts clean; set False to keep your tables
if FRESH_START:
    shutil.rmtree("lake", ignore_errors=True)

builder = (SparkSession.builder.master("local[4]").appName("final-project")
           .config("spark.driver.memory", "4g")                # 9.5M rows + Z-order need more than the 1g default
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
           .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")   # SPARK-60294
           .config("spark.ui.showConsoleProgress", "false"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()   # first run downloads the Delta jar
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

BRONZE, SILVER, QUARANTINE, GOLD = "lake/bronze/trips", "lake/silver/trips", "lake/quarantine/trips", "lake/gold/daily_borough"
zones = (spark.read.option("header", True)
         .schema("LocationID int, Borough string, Zone string, service_zone string")
         .csv("data/taxi_zone_lookup.csv"))

def table(path):
    return spark.read.format("delta").load(path)

def plan(df, mode="formatted"):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

_EXPECTED = {
 "P1": "58048ee5a018b589",
 "P2": "f2b7958e3a4f6e84",
 "P3": "b995ac3515d88da6",
 "P5": "b971d0e1bff02e6c"
}
_RECORD = os.environ.get("LAB_RECORD")

def _norm(v):
    if isinstance(v, bool) or v is None: return v
    if isinstance(v, int): return v
    if isinstance(v, float): return round(v, 2)
    if isinstance(v, str): return v.strip()
    if isinstance(v, dict): return {str(k): _norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
    if isinstance(v, (list, tuple)): return [_norm(x) for x in v]
    if hasattr(v, "item"): return _norm(v.item())
    if type(v).__name__ == "Decimal": return round(float(v), 2)
    raise TypeError(f"check() needs plain Python values, got {type(v).__name__}")

def check(task, value):
    normal = _norm(value)
    if _RECORD:
        rec = json.load(open(_RECORD)) if os.path.exists(_RECORD) else {}
        rec[task] = normal; json.dump(rec, open(_RECORD, "w"))
    ok = hashlib.sha256(json.dumps(normal).encode()).hexdigest()[:16] == _EXPECTED.get(task)
    print(f"✅ {task} correct" if ok else f"❌ {task} not yet: compare your code with the requirements again")

def check_true(task, condition, hint=""):
    print(f"✅ {task} correct" if condition else f"❌ {task} not yet. {hint}")


## Milestone 1: bronze

**P1. Write `ingest_month(month)` that appends one raw monthly file to the Delta table at `BRONZE`.**
* read `data/yellow_tripdata_{month}.parquet` as it is, and add *source_file* (the file name) and *ingested_at* (the load time)
* it must be **idempotent**: if that file is already in bronze, do nothing
* load January and February (March comes in milestone 5)

Put the number of bronze rows in *bronze_rows*.

In [ ]:
check("P1", bronze_rows)
before = table(BRONZE).count(); ingest_month("2024-01")
check_true("P1b", table(BRONZE).count() == before, "loading January again must not add rows")

## Milestone 2: silver and quarantine

**P2. Write `build_silver()` that turns bronze into the clean Delta table `SILVER`, and writes rejected rows to `QUARANTINE`.**
* reject a row, with a *reason* column, using the first rule it breaks, in this order:
  1. `pickup outside file month`: the pickup's year-month differs from the month in *source_file*
  2. `negative fare`: *fare_amount* < 0
  3. `no distance`: *trip_distance* is not > 0
  4. `dropoff not after pickup`
* drop exact duplicates of the original 19 columns
* add *trip_id* = `sha2` of the original 19 columns (hint: `F.to_json(F.struct(...))`), *pickup_date* and *duration_min*
* store money columns (*fare_amount*, *tip_amount*, *total_amount*) as `decimal(10,2)`
* silver is loaded with a **MERGE on trip_id** (insert only new trips), so running it twice changes nothing
* silver must have **Change Data Feed** enabled from its first version (you need it in milestone 5)

Put the silver row count in *silver_rows* and the quarantine counts per reason in *reasons* (`{reason: count}`).

In [ ]:
check("P2", [silver_rows, reasons])
s = table(SILVER)
check_true("P2b", s.select("trip_id").distinct().count() == s.count(), "trip_id must be unique")
build_silver()
n_before = table(SILVER).count(); build_silver()
check_true("P2c", table(SILVER).count() == n_before, "a second build_silver() must not add rows")
check_true("P2d", dict(s.dtypes)["total_amount"].startswith("decimal"), "money columns must be decimal")

## Milestone 3: gold

**P3. Write `gold_from(silver_df)` and build the Delta table `GOLD`: one row per *pickup_date* and pickup *Borough*.**
* columns: *pickup_date*, *Borough*, *trips* (count), *revenue* (sum of *total_amount*), *tips* (sum of *tip_amount*)
* join *zones* with a **broadcast** join (the check reads your plan)
* save the silver version you built it from in *gold_built_from* (you need it in milestone 5)

Put the number of gold rows in *gold_rows* and the total revenue (a float, 2 decimals) in *gold_revenue*.

In [ ]:
check("P3", [gold_rows, gold_revenue])
g = table(GOLD)
check_true("P3b", g.agg(F.sum("trips")).first()[0] == table(SILVER).count(), "gold trips must add up to the silver row count")
check_true("P3c", "BroadcastHashJoin" in plan(gold_from(table(SILVER)), "simple"), "zones should be broadcast")

## Milestone 4: performance

**P4. Make lookups by pickup zone cheap on silver.**
* set `spark.databricks.delta.optimize.maxFileSize` to 32 MB (in bytes) so compaction writes several files
* `OPTIMIZE` silver with `ZORDER BY (PULocationID)`
* compare how many files a `PULocationID = 132` (JFK Airport) filter reads, with `df.inputFiles()`, against all silver files

Put `(files_for_jfk, all_files)` in *files*.

In [ ]:
check_true("P4", files[0] < files[1] / 2, "after Z-ordering, the JFK filter should read less than half of the files")
last = DeltaTable.forPath(spark, SILVER).history(1).first()
check_true("P4b", last["operation"] == "OPTIMIZE" and "PULocationID" in str(last["operationParameters"]), "the last silver commit should be OPTIMIZE with zOrderBy PULocationID")

## Milestone 5: March arrives

**P5. March has arrived. Update gold incrementally, without recomputing it from all of silver.**
* `ingest_month("2024-03")`, then `build_silver()`
* read only silver's **change data feed** since *gold_built_from* (inserted rows)
* aggregate those rows with `gold_from` and **MERGE** them into gold: add to *trips*, *revenue* and *tips* for existing (date, borough) rows, insert new ones

Put the new gold row count in *gold_rows_after* and total revenue in *gold_revenue_after*.

In [ ]:
check("P5", [gold_rows_after, gold_revenue_after])
from pyspark.testing import assertDataFrameEqual
try:
    assertDataFrameEqual(table(GOLD), gold_from(table(SILVER)))
    check_true("P5b", True)
except AssertionError as e:
    check_true("P5b", False, "incremental gold must equal gold recomputed from all of silver")

## Milestone 6: report (not checked)

**P6. In a markdown cell, write half a page for your team:**
* how many rows each rule rejected, and one example row per reason
* the plan of the gold build: join strategy, number of shuffles, what was pushed down
* what OPTIMIZE and ZORDER changed in the `_delta_log`, and what `VACUUM` would delete now
* one thing you would change before running this every month in production

Stretch: force a sort-merge join for gold (broadcast off, AQE off), find the Manhattan straggler in the Spark UI, then fix it with AQE skew handling.

In [ ]:
print("Write your notes in a markdown cell above.")

---
Done? Put the notebook in your own GitHub repository with a short README: it is your first Spark portfolio project.